# Phrase Analysis

Analysis of participant phrases across all 7 topics and 505 images.

### Data Sources
- **Sheet 1160817631** — Pivot table: topic × phrase × image count per VisType
- **Sheet 1429789806** — Per-image data: VisType, NormalizedVC, humanCuratedPhrases, Topics, originalSentiments (all '; '-separated)

### Goals
1. Stacked horizontal bar chart: top phrases by total image count, bars colored by NormalizedVC interval (0.2 bins)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

spreadsheet_id = '1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc'

VisTypes = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

topic_names = [
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
    'Immediacy / Cognitive Load',
]

VC_BINS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
VC_LABELS = ['0.0–0.2', '0.2–0.4', '0.4–0.6', '0.6–0.8', '0.8–1.0']
VC_COLORS = ['#4393c3', '#92c5de', '#f4a582', '#d6604d', '#b2182b']


## Step 0: Load Data


In [2]:
# Sheet 1: pivot table — one row per (topic, phrase), columns = VisType counts
sheet1_id = '1160817631'
df_pivot = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet1_id}&format=csv'
)
print(df_pivot.shape)
print(df_pivot.columns.tolist())
df_pivot.head()


(405, 13)
['topic', 'phrase', 'sentiment', 'num_images', 'Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']


,topic,phrase,sentiment,num_images,Area,Bar,Cont.-ColorPatn,Glyph,Grid,Line,Node-link,Point,Text
0,Aesthetics Uncertainty,arbitrary imaging,(+),1,0,0,0,0,0,0,1,0,0
1,Aesthetics Uncertainty,convoluted composition,(+),2,0,0,0,0,1,0,0,1,0
2,Aesthetics Uncertainty,distracting/confusing/unclear,(+),23,3,1,3,4,2,4,1,2,1
3,Aesthetics Uncertainty,feeling strange,(+),1,0,0,0,0,0,0,1,0,0
4,Aesthetics Uncertainty,inconsistent,(+),1,0,0,0,0,0,0,0,0,1


In [3]:
# Sheet 2: per-image data — VisType, NormalizedVC, humanCuratedPhrases, Topics, originalSentiments
# OLD: sheet2_id = '1390591889'  # had 7 separate topic columns with sentiment-prefixed phrases
sheet2_id = '1429789806'  # new format: humanCuratedPhrases ('; '-separated), Topics, originalSentiments
df_images = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet2_id}&format=csv'
)
print(df_images.shape)
print(df_images.columns.tolist())
df_images.head()


(520, 17)
['image', 'imageName', 'imageURL', 'VisType', 'NormalizedVC', 'context', 'rawUserComments', 'originalPhrases', 'humanCuratedPhrases', 'finalPhrases', 'Topics', 'SubTopics', 'originalSentiments', 'originalStems', 'originalStemPOS', 'objectWords', 'actionWords']


,image,imageName,imageURL,VisType,NormalizedVC,context,rawUserComments,originalPhrases,humanCuratedPhrases,finalPhrases,Topics,SubTopics,originalSentiments,originalStems,originalStemPOS,objectWords,actionWords
0,NaN,wsj21.png,https://raw.githubusercontent.com/c109363/Expe...,Area,0.63,1.0,The other image has visuals to make it easier ...,different visual elements; allow to analyze th...,several elements; easier to interpret/read/und...,easy/hard to interpret,Immediacy / Cognitive Load; Visual Encoding Cl...,easy/hard to interpret,(-); (-); (-),differ; visual; element; allow; analyze; datum...,ADJ; ADJ; NOUN; VERB; VERB; NOUN; ADJ; VERB,element; datum,allow; analyze; understand
1,NaN,InfoVisJ.1009.9.png,https://raw.githubusercontent.com/c109363/Expe...,Area,0.73,0.0,There is no key for the data. The different co...,no key for the data; different colors; differe...,lack of/not enough axis labels/legend/annotati...,color contrast/clarity; color variety/shading;...,"Color, Symbol, and Texture Details; Semantics ...",color contrast/clarity; color variety/shading;...,(+); (+) (-); (+); (+),key; datum; differ; color; confuse; shade; change,NOUN; NOUN; ADJ; NOUN; VERB; NOUN; NOUN,key; datum; color; shade; change,confuse
2,NaN,visMost777.png,https://raw.githubusercontent.com/c109363/Expe...,Area,0.64,1.0,The image changes over something but not sure ...,not sure; physical dimension; meanings of colo...,2D/3D; unfamiliar concepts/patterns; unclear c...,2D/3D; color contrast/clarity; domain-specific...,"Color, Symbol, and Texture Details; Schema",2D/3D; color contrast/clarity; domain-specific...,(+); (+); (+),sure; physical; dimension; mean; color; define,ADJ; ADJ; NOUN; NOUN; NOUN; VERB,dimension; meaning; color,define
3,NaN,v484_n7395_11_f2.png,https://raw.githubusercontent.com/c109363/Expe...,Area,0.70,0.0,There is a multitude of differing information ...,multitude of differing information; color keys...,diverse information; color keys; hard to compa...,color variety/shading; easy/hard to interpret;...,"Color, Symbol, and Texture Details; Data Densi...",color variety/shading; easy/hard to interpret;...,(+); (+); (+),multitude; differ; information; color; key; ha...,NOUN; VERB; NOUN; NOUN; NOUN; ADV; VERB,multitude; information; color; key,differ; ascertain
4,NaN,vis3.png,https://raw.githubusercontent.com/c109363/Expe...,Area,0.70,0.0,Multiple colour schemes and shade variation. D...,shape varation; Multiple color schemes; Differ...,shape variety; different font/word sizes/struc...,color variety/shading; shapes and lines; take ...,"Color, Symbol, and Texture Details; Immediacy ...",color variety/shading; shapes and lines; take ...,(+); (+); (+); (+),shape; varation; multiple; color; scheme; diff...,PROPN; PROPN; ADJ; NOUN; NOUN; VERB; ADJ; NOUN...,shape; varation; color; scheme; size,differ; take; understand


## Step 1: Build Phrase → NormalizedVC Counts


In [4]:
def parse_semicolon_list(cell_value):
    """Split a '; '-separated cell into a list of stripped, non-empty strings."""
    if not isinstance(cell_value, str) or cell_value.strip() == '':
        return []
    return [p.strip() for p in cell_value.split(';') if p.strip()]

# Topic list (same 7 topics as before)
topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

# Build flat (phrase, NormalizedVC) records — one record per phrase per image
# New sheet: phrases in 'humanCuratedPhrases', topics in 'Topics' (both '; '-separated)
records = []
for _, row in df_images.iterrows():
    vc = row['NormalizedVC']
    if pd.isna(vc):
        continue
    phrases = parse_semicolon_list(row.get('humanCuratedPhrases', ''))
    for phrase in phrases:
        records.append({'phrase': phrase, 'NormalizedVC': float(vc)})

df_records = pd.DataFrame(records)
print(f'Total phrase-image pairs: {len(df_records)}')
print(f'Unique phrases: {df_records["phrase"].nunique()}')

# Bin NormalizedVC into intervals
df_records['vc_bin'] = pd.cut(
    df_records['NormalizedVC'],
    bins=VC_BINS,
    labels=VC_LABELS,
    right=False
)

# Count per (phrase, vc_bin)
df_counts = (
    df_records
    .groupby(['phrase', 'vc_bin'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=VC_LABELS, fill_value=0)
)
df_counts['total'] = df_counts[VC_LABELS].sum(axis=1)
# Sort ascending so highest-count phrase is at top of horizontal bar chart
df_counts = df_counts.sort_values('total', ascending=True)

print(f'\nPhrases in counts table: {len(df_counts)}')
df_counts.tail()


Total phrase-image pairs: 1843
Unique phrases: 400

Phrases in counts table: 400


vc_bin,0.0–0.2,0.2–0.4,0.4–0.6,0.6–0.8,0.8–1.0,total
phrase,,,,,,
"domain-specific concepts (e.g., chemical, biology, map)",0,2,16,21,2,41
easy to interpret/read/understand,0,16,26,11,2,55
much/more data/info/info spread,0,4,26,24,3,57
more charts/points/lines/shapes/elements,0,1,26,40,11,78
color variety/arrangement/distribution,0,4,29,46,7,86


In [5]:
# Distribution of phrase total counts — helps decide MIN_COUNT threshold
for threshold in [1, 2, 3, 4, 5, 10]:
    n = (df_counts['total'] >= threshold).sum()
    print(f'  >= {threshold:2d} images: {n:4d} phrases')


  >=  1 images:  400 phrases
  >=  2 images:  200 phrases
  >=  3 images:  136 phrases
  >=  4 images:  107 phrases
  >=  5 images:   87 phrases
  >= 10 images:   41 phrases


In [6]:
# ── Phrase vocabulary comparison ──────────────────────────────────────────────
# Pivot table (Sheet 1): rows are (topic, phrase, sentiment) triples
print('=== Sheet 1 – df_pivot ===')
print(f'  Rows (topic×phrase×sentiment combos): {len(df_pivot)}')
print(f'  Unique phrases (ignoring topic & sentiment): {df_pivot["phrase"].nunique()}')
print(f'  Phrases appearing with BOTH (+) and (-):',
      df_pivot.groupby('phrase')['sentiment'].nunique().gt(1).sum())

print()
print('=== Sheet 2 – df_images (per-image, after stripping sentiment) ===')
print(f'  Unique phrase strings (sentiment already stripped): {df_records["phrase"].nunique()}')

# How many of the 555 are NOT in the pivot's vocabulary?
pivot_vocab = set(df_pivot['phrase'].str.strip())
img_vocab   = set(df_records['phrase'].str.strip())
only_in_images = img_vocab - pivot_vocab
only_in_pivot  = pivot_vocab - img_vocab
print(f'  Phrases only in df_images (not in pivot): {len(only_in_images)}')
print(f'  Phrases only in df_pivot  (not in images): {len(only_in_pivot)}')
print(f'  Phrases in both: {len(img_vocab & pivot_vocab)}')


=== Sheet 1 – df_pivot ===
  Rows (topic×phrase×sentiment combos): 405
  Unique phrases (ignoring topic & sentiment): 400
  Phrases appearing with BOTH (+) and (-): 2

=== Sheet 2 – df_images (per-image, after stripping sentiment) ===
  Unique phrase strings (sentiment already stripped): 400
  Phrases only in df_images (not in pivot): 0
  Phrases only in df_pivot  (not in images): 0
  Phrases in both: 400


## Step 2: Stacked Horizontal Bar Chart


In [13]:
TOP_N = 40

df_plot = df_counts.drop(columns='total')
totals = df_counts['total']

# Keep only top N phrases by total count
top_phrases = totals.sort_values(ascending=False).head(TOP_N).index
df_plot = df_plot.loc[top_phrases]
# Sort ascending so highest-count phrase is at top of horizontal bar chart
sort_order = totals.loc[df_plot.index].sort_values(ascending=True).index
df_plot = df_plot.loc[sort_order]
totals = totals.loc[sort_order]

fig_height = max(6, len(df_plot) * 0.22)
fig, ax = plt.subplots(figsize=(12, fig_height))

left = np.zeros(len(df_plot))
for label, color in zip(VC_LABELS, VC_COLORS):
    values = df_plot[label].values
    ax.barh(df_plot.index, values, left=left, color=color, label=label, height=0.75)
    left += values

# Label total count at the end of each bar
for i, (phrase, total) in enumerate(totals.items()):
    ax.text(total + 0.3, i, str(total), va='center', ha='left', fontsize=7)

ax.set_xlabel('Number of images', fontsize=11)
ax.set_title(
    f'Top {TOP_N} phrases by NormalizedVC interval (of {len(df_counts)} total)',
    fontsize=12, pad=12
)
ax.tick_params(axis='y', labelsize=7.5)
ax.set_xlim(left=0, right=totals.max() + 8)
ax.set_ylim(-0.5, len(df_plot) - 0.5)
ax.spines[['right', 'top']].set_visible(False)

# Legend — VC intervals ordered low→high
handles = [mpatches.Patch(color=c, label=l) for c, l in zip(VC_COLORS, VC_LABELS)]
ax.legend(handles=handles, title='NormalizedVC', loc='lower right',
          fontsize=9, title_fontsize=9, framealpha=0.8)

plt.tight_layout()
out_path = 'figures/phrase_vc_distribution.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_path}')

# Save underlying data to CSV (sorted descending for readability)
csv_path = 'figures/phrase_vc_distribution.csv'
df_counts.sort_values('total', ascending=False).to_csv(csv_path)
print(f'Saved → {csv_path}')

Saved → figures/phrase_vc_distribution.png
Saved → figures/phrase_vc_distribution.csv


In [14]:
# ── Build per-topic phrase records (df_rt used downstream) ────────────────────
records_with_topic = []
for _, row in df_images.iterrows():
    vc = row['NormalizedVC']
    if pd.isna(vc):
        continue
    phrases = parse_semicolon_list(row.get('humanCuratedPhrases', ''))
    topics = parse_semicolon_list(row.get('Topics', ''))
    for phrase in phrases:
        for topic in topics:
            records_with_topic.append({
                'topic':        topic,
                'phrase':       phrase,
                'NormalizedVC': float(vc),
            })

df_rt = pd.DataFrame(records_with_topic)
df_rt['vc_bin'] = pd.cut(df_rt['NormalizedVC'], bins=VC_BINS, labels=VC_LABELS, right=False)
print(f'Per-topic phrase records: {len(df_rt)}')

Per-topic phrase records: 6712


In [15]:
# ── Build per-VisType phrase records (df_rv used downstream) ──────────────────
records_with_vt = []
for _, row in df_images.iterrows():
    vc = row['NormalizedVC']
    vt = row['VisType']
    if pd.isna(vc) or pd.isna(vt):
        continue
    for phrase in parse_semicolon_list(row.get('humanCuratedPhrases', '')):
        records_with_vt.append({
            'VisType':      str(vt).strip(),
            'phrase':       phrase,
            'NormalizedVC': float(vc),
        })

df_rv = pd.DataFrame(records_with_vt)
df_rv['vc_bin'] = pd.cut(df_rv['NormalizedVC'], bins=VC_BINS, labels=VC_LABELS, right=False)

unknown_vts = set(df_rv['VisType'].unique()) - set(VisTypes)
if unknown_vts:
    print(f'Ignoring unrecognised VisType values: {sorted(unknown_vts)}')
print(f'Per-VisType phrase records: {len(df_rv)}')

Ignoring unrecognised VisType values: ['Schematic', 'Table']
Per-VisType phrase records: 1843


## Step 3: Diverging Bar Chart — Phrases with Both (+) and (−) Sentiment


In [16]:
# Build records keeping sentiment
# New sheet: phrases in 'humanCuratedPhrases', sentiments in 'originalSentiments' (both '; '-separated, 1:1)
records_sent = []
for _, row in df_images.iterrows():
    vc = row['NormalizedVC']
    if pd.isna(vc):
        continue
    phrases = parse_semicolon_list(row.get('humanCuratedPhrases', ''))
    sentiments = parse_semicolon_list(row.get('originalSentiments', ''))
    for phrase, sent_raw in zip(phrases, sentiments):
        if not phrase:
            continue
        # sent_raw may be "(+)", "(-)", or "(+) (-)" for dual sentiment
        sent_tokens = [s.strip() for s in sent_raw.split() if s.strip() in ('(+)', '(-)')]
        if not sent_tokens:
            sent_tokens = ['(+)']  # default to positive
        for token in sent_tokens:
            sentiment = '+' if token == '(+)' else '-'
            records_sent.append({'phrase': phrase, 'sentiment': sentiment, 'NormalizedVC': float(vc)})

df_sent = pd.DataFrame(records_sent)
df_sent['vc_bin'] = pd.cut(df_sent['NormalizedVC'], bins=VC_BINS, labels=VC_LABELS, right=False)

# Keep only phrases that appear with BOTH sentiments
both = df_sent.groupby('phrase')['sentiment'].nunique()
dual_phrases = both[both == 2].index
df_dual = df_sent[df_sent['phrase'].isin(dual_phrases)]
print(f'Phrases with both (+) and (-): {len(dual_phrases)}')

# Count per (phrase, sentiment, vc_bin)
counts_dual = (
    df_dual
    .groupby(['phrase', 'sentiment', 'vc_bin'], observed=True)
    .size()
    .unstack('vc_bin', fill_value=0)
    .reindex(columns=VC_LABELS, fill_value=0)
)

pos = counts_dual.xs('+', level='sentiment')   # positive counts (right)
neg = counts_dual.xs('-', level='sentiment')   # negative counts (left)

# Align indices (both should be identical but guard just in case)
all_phrases = pos.index.union(neg.index)
pos = pos.reindex(all_phrases, fill_value=0)
neg = neg.reindex(all_phrases, fill_value=0)

# Sort by total positive count descending (ascending for barh → highest at top)
sort_order = pos.sum(axis=1).sort_values(ascending=True).index
pos = pos.loc[sort_order]
neg = neg.loc[sort_order]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_height = max(4, len(pos) * 0.22)
fig, ax = plt.subplots(figsize=(14, fig_height))

# Right side: (+) — plot each VC bin stacked left→right from 0
left_pos = np.zeros(len(pos))
for label, color in zip(VC_LABELS, VC_COLORS):
    vals = pos[label].values
    ax.barh(pos.index, vals, left=left_pos, color=color, height=0.7)
    left_pos += vals

# Left side: (-) — plot mirrored (negative x values), stacked right→left from 0
left_neg = np.zeros(len(neg))
for label, color in zip(VC_LABELS, VC_COLORS):
    vals = neg[label].values
    ax.barh(neg.index, -vals, left=-left_neg, color=color, height=0.7)
    left_neg += vals

# Count labels at bar ends
pos_totals = pos.sum(axis=1)
neg_totals = neg.sum(axis=1)
for i, phrase in enumerate(pos.index):
    ax.text( pos_totals[phrase] + 0.3, i,  str(pos_totals[phrase]), va='center', ha='left',  fontsize=7)
    ax.text(-neg_totals[phrase] - 0.3, i, str(neg_totals[phrase]),  va='center', ha='right', fontsize=7)

max_val = max(pos_totals.max(), neg_totals.max())
ax.set_xlim(-max_val - 6, max_val + 6)
ax.set_ylim(-0.5, len(pos) - 0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('← (−) negative          Number of images          (+) positive →', fontsize=10)
ax.set_title(
    f'Phrases with both (+) and (−) sentiment (n={len(pos)}, all topics & VisTypes)',
    fontsize=12, pad=12
)
ax.tick_params(axis='y', labelsize=7.5)
ax.tick_params(axis='x', labelsize=8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: str(abs(int(x)))))
ax.spines[['right', 'top']].set_visible(False)

handles = [mpatches.Patch(color=c, label=l) for c, l in zip(VC_COLORS, VC_LABELS)]
ax.legend(handles=handles, title='NormalizedVC', loc='lower right',
          fontsize=9, title_fontsize=9, framealpha=0.8)

plt.tight_layout()
out_path = 'figures/phrase_diverging_sentiment.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_path}')


Phrases with both (+) and (-): 131
Saved → figures/phrase_diverging_sentiment.png


## Step 4: Phrase Co-occurrence Across VisTypes


### Step 4a: Phrase × VisType Heatmap & Jaccard Similarity

In [18]:
import matplotlib.colors as mcolors

TOP_N = 30   # top phrases by total image count

# ── phrase × VisType count matrix (canonical VisTypes only) ───────────────────
df_rv_canon = df_rv[df_rv['VisType'].isin(VisTypes)]

pv_counts = (
    df_rv_canon
    .groupby(['phrase', 'VisType'], observed=True)
    .size()
    .unstack('VisType', fill_value=0)
    .reindex(columns=VisTypes, fill_value=0)
)

# Number of images per VisType (from raw df_images, canonical only)
vt_image_counts = (
    df_images[df_images['VisType'].isin(VisTypes)]
    ['VisType'].value_counts()
    .reindex(VisTypes)
)
print('Images per VisType:')
print(vt_image_counts.to_string())

# Normalise: phrase prevalence = count / total images of that VisType
pv_norm = pv_counts.div(vt_image_counts, axis=1)   # values in [0, 1]

# Filter to top N phrases by overall total
top_phrases = df_counts['total'].nlargest(TOP_N).index

# Sort columns (VisTypes) by total count descending (high on left)
vt_col_order = pv_counts.reindex(top_phrases).sum().sort_values(ascending=False).index.tolist()
# Sort rows (phrases) by total count descending (high on top)
pv_top = pv_norm.reindex(top_phrases)[vt_col_order]
row_order = pv_counts.reindex(top_phrases).sum(axis=1).sort_values(ascending=False).index
pv_top = pv_top.reindex(row_order)

# ── Plot 1: Phrase × VisType heatmap ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, TOP_N * 0.28 + 1.5))
im = ax.imshow(pv_top.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=pv_top.values.max())

ax.set_xticks(range(len(vt_col_order)))
ax.set_xticklabels(vt_col_order, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(pv_top.index, fontsize=8)
ax.set_title(f'Phrase prevalence by VisType (top {TOP_N} phrases)', fontsize=12, pad=12)

# Annotate cells with raw counts
pv_top_raw = pv_counts.reindex(row_order)[vt_col_order]
for r in range(TOP_N):
    for c, vt in enumerate(vt_col_order):
        val = pv_top_raw.iloc[r, c]
        if val > 0:
            ax.text(c, r, str(val), ha='center', va='center', fontsize=6.5,
                    color='white' if pv_top.iloc[r, c] > pv_top.values.max() * 0.6 else 'black')

plt.colorbar(im, ax=ax, label='Prevalence (count / VisType images)', shrink=0.6)
plt.tight_layout()
plt.savefig('figures/phrase_vistype_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved → figures/phrase_vistype_heatmap.png')

# ── Plot 2: VisType × VisType Jaccard similarity ──────────────────────────────
# Binary: phrase present in VisType if count >= 1
binary = (pv_counts > 0).astype(int)   # shape: (404 phrases) × (9 VisTypes)

n_vt = len(VisTypes)
jaccard = np.zeros((n_vt, n_vt))
for i in range(n_vt):
    for j in range(n_vt):
        a = binary.iloc[:, i].values
        b = binary.iloc[:, j].values
        intersection = (a & b).sum()
        union        = (a | b).sum()
        jaccard[i, j] = intersection / union if union > 0 else 0.0

fig2, ax2 = plt.subplots(figsize=(7, 6))
im2 = ax2.imshow(jaccard, cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(n_vt)); ax2.set_xticklabels(VisTypes, rotation=35, ha='right', fontsize=9)
ax2.set_yticks(range(n_vt)); ax2.set_yticklabels(VisTypes, fontsize=9)
ax2.set_title('VisType × VisType phrase overlap (Jaccard)', fontsize=12, pad=12)
for i in range(n_vt):
    for j in range(n_vt):
        ax2.text(j, i, f'{jaccard[i, j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if jaccard[i, j] > 0.6 else 'black')
plt.colorbar(im2, ax=ax2, label='Jaccard similarity', shrink=0.8)
plt.tight_layout()
plt.savefig('figures/vistype_jaccard_similarity.png', dpi=150, bbox_inches='tight')
plt.close(fig2)
print('Saved → figures/vistype_jaccard_similarity.png')

Images per VisType:
VisType
Area               66
Bar                52
Cont.-ColorPatn    42
Glyph              64
Grid               65
Line               48
Node-link          66
Point              58
Text               51
Saved → figures/phrase_vistype_heatmap.png
Saved → figures/vistype_jaccard_similarity.png


### Step 4b: UpSet Plot — Phrase Co-occurrence Across VisTypes


In [21]:

# Step 4b: UpSet plot – phrase co-occurrence across VisTypes
import numpy as np
import matplotlib.gridspec as gridspec

# ── Build phrase × VisType count matrix ───────────────────────────────────────
df_rv_canon = df_rv[df_rv['VisType'].isin(VisTypes)]
pv_counts = (
    df_rv_canon
    .groupby(['phrase', 'VisType'], observed=True)
    .size()
    .unstack('VisType', fill_value=0)
    .reindex(columns=VisTypes, fill_value=0)
)

binary = (pv_counts > 0)

# Total phrases per VisType (set size)
set_sizes = binary.sum()

# Sort VisTypes from least to most phrases
sorted_vts = set_sizes.sort_values().index.tolist()
n_sets = len(sorted_vts)

# Membership tuple per phrase → count phrases sharing the same exact combo
membership_tuples = binary.apply(
    lambda row: tuple(sorted(vt for vt in row.index if row[vt])), axis=1
)
combo_counts = membership_tuples.value_counts()

# Keep intersections with ≥ 2 phrases, top 40 by count
min_size = 2
top_n    = 40
top_combos = (
    combo_counts[combo_counts >= min_size]
    .sort_values(ascending=False)
    .head(top_n)
)
n_combos = len(top_combos)

# y-position per VisType (lowest set size at bottom, highest at top)
vt_y = {vt: i for i, vt in enumerate(sorted_vts)}

# ── Layout ────────────────────────────────────────────────────────────────────
set_bar_w   = 2.5
gap_w       = 0.4                               # spacing for labels
combo_bar_w = max(12, n_combos * 0.45)
matrix_h    = n_sets * 0.40
bar_h       = 2.5

fig = plt.figure(figsize=(set_bar_w + gap_w + combo_bar_w, bar_h + matrix_h))
gs  = gridspec.GridSpec(
    2, 2,
    figure=fig,
    width_ratios=[set_bar_w, combo_bar_w],
    height_ratios=[bar_h, matrix_h],
    hspace=0.05,
    wspace=0.18,
)

ax_inter  = fig.add_subplot(gs[0, 1])          # intersection bars  (top-right)
ax_size   = fig.add_subplot(gs[1, 0])          # set-size bars      (bottom-left)
ax_matrix = fig.add_subplot(gs[1, 1])          # dot matrix         (bottom-right)
ax_corner = fig.add_subplot(gs[0, 0])          # empty corner
ax_corner.axis('off')

# ── Intersection bars (top-right) ────────────────────────────────────────────
bar_x = np.arange(n_combos)
bars  = ax_inter.bar(bar_x, top_combos.values, color='#333333', width=0.55,
                     edgecolor='#333333', linewidth=0.4)
for bar, val in zip(bars, top_combos.values):
    ax_inter.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(val), ha='center', va='bottom', fontsize=9,
    )
ax_inter.set_xticks([])
ax_inter.set_ylabel('')
ax_inter.set_xlim(-0.5, n_combos - 0.5)
ax_inter.set_ylim(bottom=0)
ax_inter.text(0.2, 0.92, 'Intersection Size', transform=ax_inter.transAxes,
              ha='center', va='top', fontsize=12)
for sp in ['top', 'right', 'bottom']:
    ax_inter.spines[sp].set_visible(False)

# ── Dot matrix (bottom-right) ────────────────────────────────────────────────
ax_matrix.set_xlim(-0.5, n_combos - 0.5)
ax_matrix.set_ylim(n_sets - 0.5, -0.5)          # inverted to match set-size bars

# Alternating row shading
for idx, vt in enumerate(sorted_vts):
    if idx % 2 == 0:
        ax_matrix.axhspan(vt_y[vt] - 0.5, vt_y[vt] + 0.5,
                          color='#f0f0f0', zorder=0)

# Background light dots (inactive)
for vt in sorted_vts:
    ax_matrix.scatter(
        bar_x, [vt_y[vt]] * n_combos,
        color='#e0e0e0', s=50, zorder=2, linewidths=0,
    )

# Active dots + connecting lines
for j, (combo, _) in enumerate(top_combos.items()):
    active_ys = sorted(vt_y[vt] for vt in combo if vt in vt_y)
    if len(active_ys) > 1:
        ax_matrix.plot(
            [j, j], [active_ys[0], active_ys[-1]],
            color='#333333', linewidth=1.2, zorder=3, solid_capstyle='round',
        )
    ax_matrix.scatter(
        [j] * len(active_ys), active_ys,
        color='#333333', s=50, zorder=4, linewidths=0,
    )

ax_matrix.set_xticks([])
ax_matrix.set_yticks([vt_y[vt] for vt in sorted_vts])
ax_matrix.set_yticklabels(sorted_vts, fontsize=11)
ax_matrix.tick_params(axis='y', length=0, pad=6)
for sp in ax_matrix.spines.values():
    sp.set_visible(False)

# ── Set-size bars (bottom-left) — no VisType labels ──────────────────────────
set_y_vals = [vt_y[vt] for vt in sorted_vts]
set_vals   = set_sizes[sorted_vts].values
bars_h = ax_size.barh(set_y_vals, set_vals, color='#555555', height=0.55,
                      edgecolor='#555555', linewidth=0.4)
ax_size.invert_yaxis()                          # small set sizes at top
# Count labels at bar end (left side, since x is inverted)
for yv, sv in zip(set_y_vals, set_vals):
    ax_size.text(sv + 1, yv, str(int(sv)), va='center', ha='right', fontsize=7)
ax_size.set_yticks([])                          # hide VisType labels here
ax_size.set_ylim(n_sets - 0.5, -0.5)           # match inverted direction
ax_size.set_xlim(150, 0)                        # 150 on left, 0 on right
ax_size.text(-0.03, 0.93, 'Set Size', transform=ax_size.transAxes,
             ha='left', va='top', fontsize=12)
for sp in ['top', 'right', 'left']:
    ax_size.spines[sp].set_visible(False)

plt.savefig('figures/phrase_upset_vistype.png', dpi=150, bbox_inches='tight')
plt.close(fig)

print(f"Saved figures/phrase_upset_vistype.png  ({n_combos} intersections, min {min_size} phrases each)")

Saved figures/phrase_upset_vistype.png  (40 intersections, min 2 phrases each)


### Step 4c: UpSet Plot — Phrase Co-occurrence Across Topics

In [28]:
# Step 4c: UpSet plot – phrase co-occurrence across Topics
import numpy as np
import matplotlib.gridspec as gridspec

# ── Build phrase × topic binary matrix ────────────────────────────────────────
pt_counts = (
    df_rt
    .groupby(['phrase', 'topic'], observed=True)
    .size()
    .unstack('topic', fill_value=0)
    .reindex(columns=topic_cols, fill_value=0)
)

binary_t = (pt_counts > 0)

# Total phrases per topic (set size)
set_sizes_t = binary_t.sum()

# Sort topics from least to most phrases
sorted_topics = set_sizes_t.sort_values().index.tolist()
n_sets_t = len(sorted_topics)

# Membership tuple per phrase → count phrases sharing the same exact combo
membership_tuples_t = binary_t.apply(
    lambda row: tuple(sorted(t for t in row.index if row[t])), axis=1
)
combo_counts_t = membership_tuples_t.value_counts()

# Keep intersections with ≥ 1 phrase, top 40 by count
min_size_t = 1
top_n_t    = 40
top_combos_t = (
    combo_counts_t[combo_counts_t >= min_size_t]
    .sort_values(ascending=False)
    .head(top_n_t)
)
n_combos_t = len(top_combos_t)

# y-position per topic
topic_y = {t: i for i, t in enumerate(sorted_topics)}

# ── Layout ────────────────────────────────────────────────────────────────────
set_bar_w_t   = 3.5                              # wider for longer topic labels
gap_w_t       = 0.4
combo_bar_w_t = max(8, n_combos_t * 0.45)
matrix_h_t    = n_sets_t * 0.45
bar_h_t       = 2.5

fig = plt.figure(figsize=(set_bar_w_t + gap_w_t + combo_bar_w_t, bar_h_t + matrix_h_t))
gs  = gridspec.GridSpec(
    2, 2,
    figure=fig,
    width_ratios=[set_bar_w_t, combo_bar_w_t],
    height_ratios=[bar_h_t, matrix_h_t],
    hspace=0.05,
    wspace=0.33,
)

ax_inter  = fig.add_subplot(gs[0, 1])
ax_size   = fig.add_subplot(gs[1, 0])
ax_matrix = fig.add_subplot(gs[1, 1])
ax_corner = fig.add_subplot(gs[0, 0])
ax_corner.axis('off')

# ── Intersection bars (top-right) ────────────────────────────────────────────
bar_x = np.arange(n_combos_t)
bars  = ax_inter.bar(bar_x, top_combos_t.values, color='#333333', width=0.55,
                     edgecolor='#333333', linewidth=0.4)
for bar, val in zip(bars, top_combos_t.values):
    ax_inter.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(val), ha='center', va='bottom', fontsize=9,
    )
ax_inter.set_xticks([])
ax_inter.set_ylabel('')
ax_inter.set_xlim(-0.5, n_combos_t - 0.5)
ax_inter.set_ylim(bottom=0)
ax_inter.text(0.35, 0.92, 'Intersection Size', transform=ax_inter.transAxes,
              ha='center', va='top', fontsize=12)
for sp in ['top', 'right', 'bottom']:
    ax_inter.spines[sp].set_visible(False)

# ── Dot matrix (bottom-right) ────────────────────────────────────────────────
ax_matrix.set_xlim(-0.5, n_combos_t - 0.5)
ax_matrix.set_ylim(n_sets_t - 0.5, -0.5)            # inverted to match set-size bars

# Alternating row shading
for idx, t in enumerate(sorted_topics):
    if idx % 2 == 0:
        ax_matrix.axhspan(topic_y[t] - 0.5, topic_y[t] + 0.5,
                          color='#f0f0f0', zorder=0)

# Background light dots (inactive)
for t in sorted_topics:
    ax_matrix.scatter(
        bar_x, [topic_y[t]] * n_combos_t,
        color='#e0e0e0', s=50, zorder=2, linewidths=0,
    )

# Active dots + connecting lines
for j, (combo, _) in enumerate(top_combos_t.items()):
    active_ys = sorted(topic_y[t] for t in combo if t in topic_y)
    if len(active_ys) > 1:
        ax_matrix.plot(
            [j, j], [active_ys[0], active_ys[-1]],
            color='#333333', linewidth=1.2, zorder=3, solid_capstyle='round',
        )
    ax_matrix.scatter(
        [j] * len(active_ys), active_ys,
        color='#333333', s=50, zorder=4, linewidths=0,
    )

ax_matrix.set_xticks([])
ax_matrix.set_yticks([topic_y[t] for t in sorted_topics])
ax_matrix.set_yticklabels(sorted_topics, fontsize=9)
ax_matrix.tick_params(axis='y', length=0, pad=6)
for sp in ax_matrix.spines.values():
    sp.set_visible(False)

# ── Set-size bars (bottom-left) ──────────────────────────────────────────────
set_y_vals_t = [topic_y[t] for t in sorted_topics]
set_vals_t   = set_sizes_t[sorted_topics].values
bars_h = ax_size.barh(set_y_vals_t, set_vals_t, color='#555555', height=0.55,
                      edgecolor='#555555', linewidth=0.4)
ax_size.invert_yaxis()
for yv, sv in zip(set_y_vals_t, set_vals_t):
    ax_size.text(sv + 1, yv, str(int(sv)), va='center', ha='right', fontsize=7)
ax_size.set_yticks([])
ax_size.set_ylim(n_sets_t - 0.5, -0.5)
max_set_t = int(np.ceil(max(set_vals_t) / 50) * 50)  # round up to nearest 50
ax_size.set_xlim(max_set_t, 0)
ax_size.text(0.01, 0.93, 'Set Size', transform=ax_size.transAxes,
             ha='left', va='top', fontsize=12)
for sp in ['top', 'right', 'left']:
    ax_size.spines[sp].set_visible(False)

plt.savefig('figures/phrase_upset_topic.png', dpi=150, bbox_inches='tight')
plt.close(fig)

print(f"Saved figures/phrase_upset_topic.png  ({n_combos_t} intersections, min {min_size_t} phrase each)")

Saved figures/phrase_upset_topic.png  (40 intersections, min 1 phrase each)


### Step 4d: UpSet Plot — Topic Co-occurrence Across Images

In [27]:
# Step 4d: UpSet plot – Topic co-occurrence across Images
import numpy as np
import matplotlib.gridspec as gridspec

# ── Build binary image × topic matrix ─────────────────────────────────────────
# 1 if the image lists that topic in the '; '-separated Topics column
img_topic_binary = pd.DataFrame(False, index=df_images.index, columns=topic_cols, dtype=bool)
for idx, row in df_images.iterrows():
    topics = parse_semicolon_list(row.get('Topics', ''))
    for t in topics:
        if t in topic_cols:
            img_topic_binary.at[idx, t] = True

# Set sizes: number of images per topic
img_set_sizes = img_topic_binary.sum()

# Sort topics by set size (fewest images → most)
sorted_img_topics = img_set_sizes.sort_values().index.tolist()
n_sets_i = len(sorted_img_topics)

# Membership tuple per image → count images sharing the same topic combo
img_membership = img_topic_binary.apply(
    lambda row: tuple(sorted(t for t in row.index if row[t])), axis=1
)
# Drop images with no topics at all
img_membership = img_membership[img_membership.apply(len) > 0]
img_combo_counts = img_membership.value_counts()

min_size_i = 1
top_n_i    = 40
top_combos_i = (
    img_combo_counts[img_combo_counts >= min_size_i]
    .sort_values(ascending=False)
    .head(top_n_i)
)
n_combos_i = len(top_combos_i)

topic_y_i = {t: i for i, t in enumerate(sorted_img_topics)}

# ── Layout ────────────────────────────────────────────────────────────────────
set_bar_w_i   = 3.5
gap_w_i       = 0.4
combo_bar_w_i = max(8, n_combos_i * 0.45)
matrix_h_i    = n_sets_i * 0.45
bar_h_i       = 2.5

fig = plt.figure(figsize=(set_bar_w_i + gap_w_i + combo_bar_w_i, bar_h_i + matrix_h_i))
gs  = gridspec.GridSpec(
    2, 2,
    figure=fig,
    width_ratios=[set_bar_w_i, combo_bar_w_i],
    height_ratios=[bar_h_i, matrix_h_i],
    hspace=0.05,
    wspace=0.32,
)

ax_inter  = fig.add_subplot(gs[0, 1])
ax_size   = fig.add_subplot(gs[1, 0])
ax_matrix = fig.add_subplot(gs[1, 1])
ax_corner = fig.add_subplot(gs[0, 0]); ax_corner.axis('off')

# ── Intersection bars (top-right) ────────────────────────────────────────────
bar_x = np.arange(n_combos_i)
bars  = ax_inter.bar(bar_x, top_combos_i.values, color='#333333', width=0.55,
                     edgecolor='#333333', linewidth=0.4)
for bar, val in zip(bars, top_combos_i.values):
    ax_inter.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(val), ha='center', va='bottom', fontsize=9,
    )
ax_inter.set_xticks([])
ax_inter.set_ylabel('')
ax_inter.set_xlim(-0.5, n_combos_i - 0.5)
ax_inter.set_ylim(bottom=0)
ax_inter.text(0.35, 0.92, 'Intersection Size', transform=ax_inter.transAxes,
              ha='center', va='top', fontsize=12)
for sp in ['top', 'right', 'bottom']:
    ax_inter.spines[sp].set_visible(False)

# ── Dot matrix (bottom-right) ────────────────────────────────────────────────
ax_matrix.set_xlim(-0.5, n_combos_i - 0.5)
ax_matrix.set_ylim(n_sets_i - 0.5, -0.5)            # inverted to match set-size bars

for idx, t in enumerate(sorted_img_topics):
    if idx % 2 == 0:
        ax_matrix.axhspan(topic_y_i[t] - 0.5, topic_y_i[t] + 0.5,
                          color='#f0f0f0', zorder=0)

for t in sorted_img_topics:
    ax_matrix.scatter(
        bar_x, [topic_y_i[t]] * n_combos_i,
        color='#e0e0e0', s=50, zorder=2, linewidths=0,
    )

for j, (combo, _) in enumerate(top_combos_i.items()):
    active_ys = sorted(topic_y_i[t] for t in combo if t in topic_y_i)
    if len(active_ys) > 1:
        ax_matrix.plot(
            [j, j], [active_ys[0], active_ys[-1]],
            color='#333333', linewidth=1.2, zorder=3, solid_capstyle='round',
        )
    ax_matrix.scatter(
        [j] * len(active_ys), active_ys,
        color='#333333', s=50, zorder=4, linewidths=0,
    )

ax_matrix.set_xticks([])
ax_matrix.set_yticks([topic_y_i[t] for t in sorted_img_topics])
ax_matrix.set_yticklabels(sorted_img_topics, fontsize=9)
ax_matrix.tick_params(axis='y', length=0, pad=6)
for sp in ax_matrix.spines.values():
    sp.set_visible(False)

# ── Set-size bars (bottom-left) ──────────────────────────────────────────────
set_y_i = [topic_y_i[t] for t in sorted_img_topics]
set_v_i = img_set_sizes[sorted_img_topics].values
ax_size.barh(set_y_i, set_v_i, color='#555555', height=0.55,
             edgecolor='#555555', linewidth=0.4)
ax_size.invert_yaxis()
for yv, sv in zip(set_y_i, set_v_i):
    ax_size.text(sv + 1, yv, str(int(sv)), va='center', ha='right', fontsize=7)
ax_size.set_yticks([])
ax_size.set_ylim(n_sets_i - 0.5, -0.5)
max_set_i = int(np.ceil(max(set_v_i) / 50) * 50)
ax_size.set_xlim(max_set_i, 0)
ax_size.text(0.25, 0.93, 'Set Size', transform=ax_size.transAxes,
             ha='left', va='top', fontsize=12)
for sp in ['top', 'right', 'left']:
    ax_size.spines[sp].set_visible(False)

plt.savefig('figures/image_upset_topic.png', dpi=150, bbox_inches='tight')
plt.close(fig)

print(f"Saved figures/image_upset_topic.png  ({n_combos_i} intersections, min {min_size_i} image each)")
print(f"Images with at least one topic: {len(img_membership)}")

Saved figures/image_upset_topic.png  (40 intersections, min 1 image each)
Images with at least one topic: 520


#### Interpretation

All 520 images have at least one topic annotation, producing **87 distinct topic combinations** (top 40 shown).

**Set sizes (left bars):** *Immediacy / Cognitive Load* is the most commonly annotated topic (259 images), followed by *Color, Symbol, and Texture Details* (230) and *Data Density / Image Clutter* (209). *Aesthetics Uncertainty* is the rarest (55 images).

**Key patterns:**

- **Single-topic images are common but not dominant.** 157 images (30%) have annotations under exactly 1 topic, 160 (31%) under 2, and 109 (21%) under 3. Only 7 images (1.3%) touch all 7 topics.
- **"Immediacy" alone is the single largest intersection** — 56 images were annotated only under *Immediacy / Cognitive Load* and no other topic, suggesting many images have comments focused purely on immediate readability.
- **The most frequent pairings** involve *Immediacy*, *Color/Symbol/Texture*, and *Data Density*:
  - *Data Density + Immediacy*: 19 images
  - *Color/Symbol + Data Density*: 17 images
  - *Immediacy + Semantics*: 15 images
  - *Immediacy + Visual Encoding*: 14 images
- **Contrast with the phrase-level Topic UpSet (Step 4c):** That plot showed 71 combinations, but most phrases belong to exactly one topic. Here we see 87 combinations because individual *images* typically elicit comments across multiple topics — even though each phrase is topic-specific, a single image often triggers phrases from 2–3+ different topics.
- **Practical takeaway:** Visualization images rarely provoke comments about just one dimension. The typical image triggers discussion across 2–3 topics simultaneously, reflecting the multifaceted nature of visual complexity perception.

### Step 4e: UpSet Plot — Topic Co-occurrence Across Images, by VisType

In [30]:
# Step 4e: UpSet plot – Topic co-occurrence across Images, segmented by VisType
import numpy as np
import matplotlib.gridspec as gridspec

TOP_K = 15       # top combos per VisType block
GAP   = 0.5      # gap (in x-units) between VisType blocks

# ── Reuse topic ordering from Step 4d ─────────────────────────────────────────
# sorted_img_topics: fewest images → most;  topic_y_i: topic → y position
n_sets_e = len(sorted_img_topics)

# ── Per-VisType data ──────────────────────────────────────────────────────────
block_data = []   # list of (vt_name, top_combos_series, block_x_start)
x_cursor = 0.0

for vt in VisTypes:
    mask = df_images['VisType'] == vt
    sub_imgs = df_images[mask]
    if len(sub_imgs) == 0:
        continue

    # Binary topic matrix for this VisType's images
    btb = pd.DataFrame(False, index=sub_imgs.index, columns=topic_cols, dtype=bool)
    for idx_r, row_r in sub_imgs.iterrows():
        topics = parse_semicolon_list(row_r.get('Topics', ''))
        for t in topics:
            if t in topic_cols:
                btb.at[idx_r, t] = True

    mem = btb.apply(lambda row: tuple(sorted(t for t in row.index if row[t])), axis=1)
    mem = mem[mem.apply(len) > 0]
    cc = mem.value_counts()
    tc = cc.sort_values(ascending=False).head(TOP_K)

    if len(tc) == 0:
        continue

    block_data.append((vt, tc, x_cursor))
    x_cursor += len(tc) + GAP

total_cols = x_cursor - GAP  # remove trailing gap

# ── Global max intersection size (for shared y-axis on bar chart) ─────────────
global_max = max(tc.values[0] for _, tc, _ in block_data)

# ── Layout ────────────────────────────────────────────────────────────────────
set_bar_w_e   = 2.5
col_w         = 0.16   # width per column in inches (tight)
combo_bar_w_e = max(12, total_cols * col_w)
matrix_h_e    = n_sets_e * 0.32
bar_h_e       = 2.5

fig = plt.figure(figsize=(set_bar_w_e + 0.4 + combo_bar_w_e, bar_h_e + matrix_h_e))
gs  = gridspec.GridSpec(
    2, 2,
    figure=fig,
    width_ratios=[set_bar_w_e, combo_bar_w_e],
    height_ratios=[bar_h_e, matrix_h_e],
    hspace=0.03,
    wspace=0.20,
)

ax_inter  = fig.add_subplot(gs[0, 1])
ax_size   = fig.add_subplot(gs[1, 0])
ax_matrix = fig.add_subplot(gs[1, 1])
ax_corner = fig.add_subplot(gs[0, 0]); ax_corner.axis('off')

# Shared x-limits
x_lo, x_hi = -0.5, total_cols - 0.5

# ── Intersection bars (top-right) ────────────────────────────────────────────
for vt, tc, x0 in block_data:
    xs = np.arange(len(tc)) + x0
    ax_inter.bar(xs, tc.values, color='#333333', width=0.55,
                 edgecolor='#333333', linewidth=0.4)
    for xi, val in zip(xs, tc.values):
        ax_inter.text(xi, val + 0.15, str(val), ha='center', va='bottom', fontsize=6)

    # VisType label centred above block
    cx = x0 + (len(tc) - 1) / 2
    ax_inter.text(cx, global_max * 1.18, vt, ha='center', va='bottom',
                  fontsize=8, fontweight='regular')

    # Separator line between blocks (left edge)
    if x0 > 0:
        sep_x = x0 - GAP * 1.0
        ax_inter.axvline(sep_x, color='#cccccc', linewidth=0.6, linestyle='--', zorder=0)

ax_inter.set_xticks([])
ax_inter.set_xlim(x_lo, x_hi)
ax_inter.set_ylim(0, global_max * 1.35)
ax_inter.set_ylabel('Intersection\nSize', fontsize=8)
ax_inter.tick_params(axis='y', labelsize=7)
for sp in ['top', 'right', 'bottom']:
    ax_inter.spines[sp].set_visible(False)

# ── Dot matrix (bottom-right) ────────────────────────────────────────────────
ax_matrix.set_xlim(x_lo, x_hi)
ax_matrix.set_ylim(n_sets_e - 0.5, -0.5)

# Alternating row shading
for idx_t, t in enumerate(sorted_img_topics):
    if idx_t % 2 == 0:
        ax_matrix.axhspan(topic_y_i[t] - 0.5, topic_y_i[t] + 0.5,
                          color='#f0f0f0', zorder=0)

for vt, tc, x0 in block_data:
    xs = np.arange(len(tc)) + x0

    # Inactive dots
    for t in sorted_img_topics:
        ax_matrix.scatter(xs, [topic_y_i[t]] * len(xs),
                          color='#e0e0e0', s=20, zorder=2, linewidths=0)

    # Active dots + lines
    for j_local, (combo, _) in enumerate(tc.items()):
        xj = x0 + j_local
        active_ys = sorted(topic_y_i[t] for t in combo if t in topic_y_i)
        if len(active_ys) > 1:
            ax_matrix.plot([xj, xj], [active_ys[0], active_ys[-1]],
                           color='#333333', linewidth=1.0, zorder=3, solid_capstyle='round')
        ax_matrix.scatter([xj] * len(active_ys), active_ys,
                          color='#333333', s=20, zorder=4, linewidths=0)

    # Separator line
    if x0 > 0:
        sep_x = x0 - GAP * 1.0
        ax_matrix.axvline(sep_x, color='#cccccc', linewidth=0.6, linestyle='--', zorder=0)

ax_matrix.set_xticks([])
ax_matrix.set_yticks([topic_y_i[t] for t in sorted_img_topics])
ax_matrix.set_yticklabels(sorted_img_topics, fontsize=7)
ax_matrix.tick_params(axis='y', length=0, pad=4)
for sp in ax_matrix.spines.values():
    sp.set_visible(False)

# ── Set-size bars (bottom-left) — overall topic set sizes ─────────────────────
set_y_e = [topic_y_i[t] for t in sorted_img_topics]
set_v_e = img_set_sizes[sorted_img_topics].values
ax_size.barh(set_y_e, set_v_e, color='#555555', height=0.55,
             edgecolor='#555555', linewidth=0.4)
ax_size.invert_yaxis()
for yv, sv in zip(set_y_e, set_v_e):
    ax_size.text(sv + 1, yv, str(int(sv)), va='center', ha='right', fontsize=6)
ax_size.set_yticks([])
ax_size.set_ylim(n_sets_e - 0.5, -0.5)
max_set_e = int(np.ceil(max(set_v_e) / 50) * 50)
ax_size.set_xlim(max_set_e, 0)
ax_size.text(0.3, 0.96, 'Set Size', transform=ax_size.transAxes,
             ha='left', va='top', fontsize=8)
ax_size.tick_params(axis='x', labelsize=7)
for sp in ['top', 'right', 'left']:
    ax_size.spines[sp].set_visible(False)

plt.savefig('figures/image_topic_upset_by_vistype.png', dpi=150, bbox_inches='tight')
plt.close(fig)

total_combos = sum(len(tc) for _, tc, _ in block_data)
print(f"Saved figures/image_topic_upset_by_vistype.png  ({total_combos} intersections across {len(block_data)} VisTypes, top {TOP_K} each)")

Saved figures/image_topic_upset_by_vistype.png  (135 intersections across 9 VisTypes, top 15 each)


## Step 5: Term Mapping — Original, AI-Curated, Human-Curated

In [31]:
# Load Sheet 1561106523
sheet3_id = '1561106523'
df_terms = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet3_id}&format=csv'
)
print(f'Shape: {df_terms.shape}')
print(f'Columns ({len(df_terms.columns)}):')
for c in df_terms.columns:
    print(f'  {c!r}')
df_terms.head(3)

Shape: (365, 35)
Columns (35):
  'Immediacy / Cognitive Load_sentiment'
  'Immediacy / Cognitive Load_count'
  'Immediacy / Cognitive Load'
  'Immediacy / Cognitive Load (AIcurated)'
  'Immediacy / Cognitive Load (HumanCuration)'
  'Data Density / Image Clutter_sentiment'
  'Data Density / Image Clutter_count'
  'Data Density / Image Clutter'
  'Data Density / Image Clutter (AIcurated)'
  'Data Density / Image Clutter (HumanCuration)'
  'Visual Encoding Clarity_sentiment'
  'Visual Encoding Clarity_count'
  'Visual Encoding Clarity'
  'Visual Encoding Clarity (AIcurated)'
  'Visual Encoding Clarity (HumanCuration)'
  'Semantics / Text Legibility_sentiment'
  'Semantics / Text Legibility_count'
  'Semantics / Text Legibility'
  'Semantics / Text Legibility (AIcurated)'
  'Semantics / Text Legibility (HumanCuration)'
  'Schema_sentiment'
  'Schema_count'
  'Schema'
  'Schema (AIcurated)'
  'Schema (HumanCuration)'
  'Color, Symbol, and Texture Details_sentiment'
  'Color, Symbol, and Tex

,Immediacy / Cognitive Load_sentiment,Immediacy / Cognitive Load_count,Immediacy / Cognitive Load,Immediacy / Cognitive Load (AIcurated),Immediacy / Cognitive Load (HumanCuration),Data Density / Image Clutter_sentiment,Data Density / Image Clutter_count,Data Density / Image Clutter,Data Density / Image Clutter (AIcurated),Data Density / Image Clutter (HumanCuration),...,"Color, Symbol, and Texture Details_sentiment","Color, Symbol, and Texture Details_count","Color, Symbol, and Texture Details","Color, Symbol, and Texture Details (AIcurated)","Color, Symbol, and Texture Details (HumanCuration)",Aesthetics Uncertainty_sentiment,Aesthetics Uncertainty_count,Aesthetics Uncertainty,Aesthetics Uncertainty (AIcurated),Aesthetics Uncertainty (HumanCuration)
0,(-),8,easy to understand,easy to understand,easy to interpret/read/understand,(+),10.0,more information,more information,much/more data/info/info spread,...,(+),20.0,more colors,many/more colors,color variety/arrangement/distribution,(+),2.0,confusing,confusing,distracting/confusing/unclear
1,(+),7,difficult to understand,difficult to understand,hard to interpret/read/understand,(+),6.0,more details,more details,more detailed/things,...,(+),14.0,colors,colors,unclear colormaps,(+),1.0,Difference of clarity,difference in clarity,distracting/confusing/unclear
2,(+),6,don't understand,don't understand,not interpretable/readable/understandable,(+),4.0,more variables,more variables,more parameters/variables,...,(-),12.0,different colors,different colors,color variety/arrangement/distribution,(+),1.0,composition is more convoluted,convoluted composition,convoluted composition


In [32]:
# Inspect sample values for one topic (first 5 non-null rows)
t = 'Immediacy / Cognitive Load'
cols = [t, f'{t} (AIcurated)', f'{t} (HumanCuration)', f'{t}_sentiment', f'{t}_count']
sub = df_terms[cols].dropna(subset=[t]).head(5)
for i, (_, row) in enumerate(sub.iterrows()):
    print(f'Row {i}:')
    for c in cols:
        print(f'  {c[:30]:30s} -> {str(row[c])[:80]}')
    print()

Row 0:
  Immediacy / Cognitive Load     -> easy to understand
  Immediacy / Cognitive Load (AI -> easy to understand
  Immediacy / Cognitive Load (Hu -> easy to interpret/read/understand
  Immediacy / Cognitive Load_sen -> (-)
  Immediacy / Cognitive Load_cou -> 8

Row 1:
  Immediacy / Cognitive Load     -> difficult to understand
  Immediacy / Cognitive Load (AI -> difficult to understand
  Immediacy / Cognitive Load (Hu -> hard to interpret/read/understand
  Immediacy / Cognitive Load_sen -> (+)
  Immediacy / Cognitive Load_cou -> 7

Row 2:
  Immediacy / Cognitive Load     -> don't understand
  Immediacy / Cognitive Load (AI -> don't understand
  Immediacy / Cognitive Load (Hu -> not interpretable/readable/understandable
  Immediacy / Cognitive Load_sen -> (+)
  Immediacy / Cognitive Load_cou -> 6

Row 3:
  Immediacy / Cognitive Load     -> hard to understand
  Immediacy / Cognitive Load (AI -> difficult to understand
  Immediacy / Cognitive Load (Hu -> hard to interpret/read/underst

In [33]:
# ── Combine terms across all topics ───────────────────────────────────────────
rows = []
for topic in topic_cols:
    col_orig = topic
    col_ai   = f'{topic} (AIcurated)'
    col_hc   = f'{topic} (HumanCuration)'
    col_sent = f'{topic}_sentiment'
    col_cnt  = f'{topic}_count'

    for _, row in df_terms.iterrows():
        orig = row.get(col_orig)
        if not isinstance(orig, str) or orig.strip() == '':
            continue
        cnt = row.get(col_cnt)
        cnt = int(cnt) if pd.notna(cnt) else 0
        rows.append({
            'original':     orig.strip(),
            'sentiment':    str(row.get(col_sent, '')).strip(),
            'AIcurated':    str(row.get(col_ai, '')).strip() if isinstance(row.get(col_ai), str) else '',
            'HumanCurated': str(row.get(col_hc, '')).strip() if isinstance(row.get(col_hc), str) else '',
            'count':        cnt,
        })

df_all = pd.DataFrame(rows)
print(f'Total term rows across all topics: {len(df_all)}')
print(f'Unique originals: {df_all["original"].nunique()}')
print(f'Unique AIcurated: {df_all["AIcurated"].nunique()}')
print(f'Unique HumanCurated: {df_all["HumanCurated"].nunique()}')
print(f'Sentiments: {df_all["sentiment"].value_counts().to_dict()}')

# ── Aggregate sentiment per unique term in each column ────────────────────────
def agg_sentiment(series):
    """Aggregate sentiment values: if both present → '(+) (-)', else single."""
    vals = set(series.dropna().unique()) - {'', 'nan'}
    if '(+)' in vals and '(-)' in vals:
        return '(+) (-)'
    elif '(+)' in vals:
        return '(+)'
    elif '(-)' in vals:
        return '(-)'
    return ''

orig_sign = df_all.groupby('original')['sentiment'].apply(agg_sentiment).rename('OriginalSign')
ai_sign   = df_all.groupby('AIcurated')['sentiment'].apply(agg_sentiment).rename('AIcuratedSign')
hc_sign   = df_all.groupby('HumanCurated')['sentiment'].apply(agg_sentiment).rename('HumanCuratedSign')

# Aggregate image counts per unique term in each column
orig_cnt = df_all.groupby('original')['count'].sum().rename('OriginalPhraseImageCount')
ai_cnt   = df_all.groupby('AIcurated')['count'].sum().rename('AIcuratedPhraseImageCount')
hc_cnt   = df_all.groupby('HumanCurated')['count'].sum().rename('HumanCuratedPhraseImageCount')

# ── Build single CSV with 3 independent column-triples side by side ───────────
df_orig_unique = pd.concat([orig_sign, orig_cnt], axis=1).reset_index()
df_orig_unique.columns = ['original', 'OriginalSign', 'OriginalPhraseImageCount']
df_orig_unique = df_orig_unique.sort_values('original').reset_index(drop=True)

df_ai_unique = pd.concat([ai_sign, ai_cnt], axis=1).reset_index()
df_ai_unique.columns = ['AIcurated', 'AIcuratedSign', 'AIcuratedPhraseImageCount']
df_ai_unique = df_ai_unique[df_ai_unique['AIcurated'] != ''].sort_values('AIcurated').reset_index(drop=True)

df_hc_unique = pd.concat([hc_sign, hc_cnt], axis=1).reset_index()
df_hc_unique.columns = ['HumanCurated', 'HumanCuratedSign', 'HumanCuratedPhraseImageCount']
df_hc_unique = df_hc_unique[df_hc_unique['HumanCurated'] != ''].sort_values('HumanCurated').reset_index(drop=True)

max_len = max(len(df_orig_unique), len(df_ai_unique), len(df_hc_unique))
df_orig_unique = df_orig_unique.reindex(range(max_len)).fillna('')
df_ai_unique   = df_ai_unique.reindex(range(max_len)).fillna('')
df_hc_unique   = df_hc_unique.reindex(range(max_len)).fillna('')

df_out = pd.concat([df_orig_unique, df_ai_unique, df_hc_unique], axis=1)

n_orig = (df_out['original'] != '').sum()
n_ai   = (df_out['AIcurated'] != '').sum()
n_hc   = (df_out['HumanCurated'] != '').sum()
print(f'\nUnique Original terms:     {n_orig}')
print(f'Unique AIcurated terms:    {n_ai}')
print(f'Unique HumanCurated terms: {n_hc}')

csv_path = 'figures/term_mapping.csv'
df_out.to_csv(csv_path, index=False)
print(f'Saved → {csv_path}')

Total term rows across all topics: 1778
Unique originals: 1708
Unique AIcurated: 875
Unique HumanCurated: 400
Sentiments: {'(+)': 1367, '(-)': 407, 'nan': 4}

Unique Original terms:     1708
Unique AIcurated terms:    875
Unique HumanCurated terms: 400
Saved → figures/term_mapping.csv
